# AdaptiveDepth Phase 5 — route-aware training
Trains a separate student across the frozen Phase 4 route library using a full-depth teacher. Start with the four-step smoke run, then set `RUN_FULL = True`. Checkpoints are resumable within the same runtime workspace.

In [ ]:
import base64, os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle/working').exists() else '/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        if str(base).startswith('/content'):
            from google.colab import userdata
            return userdata.get(name)
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nIf the repository is private, add a read-only GITHUB_TOKEN to notebook Secrets and enable notebook access.')
os.chdir(repo)
print('Repository:', repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[data]'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch, transformers
assert torch.cuda.is_available(), 'Select a GPU runtime before running Phase 5'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)

In [ ]:
# Keep False for the first run. After it completes, set True and rerun this and the next two cells.
RUN_FULL = False
config = 'configs/phase5_t4.yaml' if RUN_FULL else 'configs/phase5_smoke.yaml'
dataset = 'data/phase5_train.jsonl' if RUN_FULL else 'data/phase5_train_smoke.jsonl'
workspace = 'experiments/phase5' if RUN_FULL else 'experiments/phase5_smoke'
if RUN_FULL and not pathlib.Path(dataset).exists():
    subprocess.run(['adaptivedepth', 'prepare-phase5-data', '--output', dataset, '--max-examples', '2000', '--seed', '17'], check=True)
print('Mode:', 'FULL' if RUN_FULL else 'SMOKE')
print('Dataset:', dataset)
print('Workspace:', workspace)

In [ ]:
command = [
    'adaptivedepth', 'run-phase5',
    '--config', config,
    '--routes-file', 'configs/phase5_routes.yaml',
    '--dataset', dataset,
    '--output', workspace,
    '--resume',
]
subprocess.run(command, check=True)

In [ ]:
import json, zipfile
summary = json.loads((pathlib.Path(workspace) / 'training_summary.json').read_text())
checkpoint = pathlib.Path(summary['final_checkpoint'])
archive = pathlib.Path('phase5_student.zip' if RUN_FULL else 'phase5_smoke_student.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in checkpoint.rglob('*'):
        if path.is_file() and path.name != 'training_state.pt':
            bundle.write(path, pathlib.Path(checkpoint.name) / path.relative_to(checkpoint))
print('Completed step:', summary['final_step'])
print('Final checkpoint:', checkpoint)
print('Inference-only download:', repo / archive)
print('Keep the workspace itself if you need optimizer-state resume.')
if not RUN_FULL:
    print('Smoke training complete. Set RUN_FULL = True and rerun the final three cells.')